# 8.2 模型装配、训练与测试

在训练网络时，一般的流程是通过前向计算获得网络的输出值，再通过损失函数计算
网络误差，然后通过自动求导工具计算梯度并更新，同时间隔性地测试网络的性能。对于
这种常用的训练逻辑，可以直接通过 Keras 提供的模型装配与训练等高层接口实现，简洁
清晰。

## 8.2.1 模型装配

在 Keras 中，有 2 个比较特殊的类：keras.Model 和 keras.layers.Layer 类。其中 Layer
类是网络层的母类，定义了网络层的一些常见功能，如添加权值、管理权值列表等。
Model 类是网络的母类，除了具有 Layer 类的功能，还添加了保存模型、加载模型、训练
与测试模型等便捷功能。Sequential 也是 Model 的子类，因此具有 Model 类的所有功能。

接下来介绍 Model 及其子类的模型装配与训练功能。我们以 Sequential 容器封装的网
络为例，首先创建 5 层的全连接网络，用于 MNIST 手写数字图片识别，代码如下：

In [1]:
import tensorflow as tf
from tensorflow import keras
# 导入 Sequential 容器和常用网络层
from tensorflow.keras import Sequential, layers

tf.random.set_seed(22)

# 创建 5 层的全连接网络。最后一层不加 Softmax，交叉熵损失会在内部完成 Softmax 计算。
network = Sequential([
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(10)
])

# 输入为展平后的 MNIST 图片，shape 为 [batch_size, 28 * 28]
network.build(input_shape=(None, 28 * 28))
network.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 244,522 (955.16 KB)

 Trainable params: 244,522 (955.16 KB)

 Non-trainable params: 0 (0.00 B)

创建网络后，正常的流程是循环迭代数据集多个 Epoch，每次按批产生训练数据、前向计算，然后通过损失函数计算误差值，并反向传播自动计算梯度、更新网络参数。这一部分
逻辑由于非常通用，在 Keras 中提供了 compile()和 fit()函数方便实现上述逻辑。首先通过
compile 函数指定网络使用的优化器对象、损失函数类型，评价指标等设定，这一步称为装配。例如

In [2]:
# 导入优化器，损失函数模块
from tensorflow.keras import optimizers, losses

# 模型装配
# 采用 Adam 优化器，学习率为 0.01；采用交叉熵损失函数，包含 Softmax。
network.compile(
    optimizer=optimizers.Adam(learning_rate=0.01),
    loss=losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']  # 设置测量指标为准确率
)

在 compile()函数中指定的优化器、损失函数等参数也是我们自行训练时需要设置的参数，
并没有什么特别之处，只不过 Keras 将这部分常用逻辑内部实现了，提高开发效率。

## 8.2.2 模型训练
模型装配完成后，即可通过 fit()函数送入待训练的数据集和验证用的数据集，这一步
称为模型训练。例如：

In [3]:
# 构建训练集、验证集和测试集
from tensorflow.keras import datasets

batch_size = 128
AUTOTUNE = tf.data.experimental.AUTOTUNE


(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# 从训练集中划出 10000 个样本作为验证集，剩余样本用于训练
x_val, y_val = x_train[-10000:], y_train[-10000:]
x_train, y_train = x_train[:-10000], y_train[:-10000]


def preprocess(x, y):
    # x: [b, 28, 28] -> [b, 784]，并归一化到 [0, 1]
    x = tf.cast(x, dtype=tf.float32) / 255.
    x = tf.reshape(x, [-1, 28 * 28])
    # y: [b] -> [b, 10]，用于 CategoricalCrossentropy
    y = tf.cast(y, dtype=tf.int32)
    y = tf.one_hot(y, depth=10)
    return x, y


train_db = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_db = train_db.shuffle(10000).batch(batch_size).map(preprocess).prefetch(AUTOTUNE)

val_db = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_db = val_db.batch(batch_size).map(preprocess).prefetch(AUTOTUNE)

test_db = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_db = test_db.batch(batch_size).map(preprocess).prefetch(AUTOTUNE)

print('train:', x_train.shape, y_train.shape)
print('val:', x_val.shape, y_val.shape)
print('test:', x_test.shape, y_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
train: (50000, 28, 28) (50000,)
val: (10000, 28, 28) (10000,)
test: (10000, 28, 28) (10000,)


这里将标签转换为 one-hot 编码，是因为上面装配模型时使用了
`losses.CategoricalCrossentropy(from_logits=True)`。如果希望标签仍保持 0~9 的整数形式，
可以把损失函数换成 `losses.SparseCategoricalCrossentropy(from_logits=True)`，并去掉
`tf.one_hot()` 这一步。

In [4]:
# 指定训练集为 train_db，验证集为 val_db，训练 5 个 epochs，每 2 个 epoch 验证一次
# 返回训练轨迹信息保存在 history 对象中
history = network.fit(
    train_db,
    epochs=5,
    validation_data=val_db,
    validation_freq=2
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9154 - loss: 0.2870
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9600 - loss: 0.1394 - val_accuracy: 0.9615 - val_loss: 0.1435
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9690 - loss: 0.1096
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9740 - loss: 0.0961 - val_accuracy: 0.9654 - val_loss: 0.1359
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9746 - loss: 0.0928


其中 train_db 为 tf.data.Dataset 对象，也可以传入 Numpy Array 类型的数据；epochs 参数指
定训练迭代的 Epoch 数量；validation_data 参数指定用于验证(测试)的数据集和验证的频率
validation_freq。
运行上述代码即可实现网络的训练与验证的功能，fit 函数会返回训练过程的数据记录
history，其中 history.history 为字典对象，包含了训练过程中的 loss、测量指标等记录项，
我们可以直接查看这些训练数据，例如：

In [5]:
# 打印训练过程记录。不同版本 TensorFlow 或不同运行环境下，具体数值会略有差异。
history.history

{'accuracy': [0.9153800010681152,
  0.9599800109863281,
  0.9689599871635437,
  0.9739800095558167,
  0.9745799899101257],
 'loss': [0.28699633479118347,
  0.1394178718328476,
  0.10956903547048569,
  0.09607625752687454,
  0.09279915690422058],
 'val_accuracy': [0.9614999890327454, 0.965399980545044],
 'val_loss': [0.14347819983959198, 0.13590560853481293]}

fit()函数的运行代表了网络的训练过程，因此会消耗相当的训练时间，并在训练结束
后才返回。通过 `compile()` 和 `fit()` 的方式实现训练代码非常简洁，不过接口层级较高，
灵活性会低于自定义训练循环，是否使用需要根据任务复杂度判断。

## 8.2.3 模型测试
Model 基类除了可以便捷地完成网络装配、训练和验证，还可以直接完成预测与测试。
验证集通常用于训练过程中选择模型或调节超参数，测试集则用于在训练完成后评估最终
模型的泛化性能。

通过 `Model.predict(x)` 方法即可完成模型预测。例如，先从测试集中加载一个 batch 的
图片，再将其送入训练好的网络：

In [6]:
# 加载一个 batch 的测试数据
x, y = next(iter(test_db))
print('predict x:', x.shape)

# 模型预测，预测结果保存在 out 中
out = network.predict(x)
print('out:', out.shape)
print(out[:3])

# 将 logits 转换为类别编号，便于和真实标签对比
pred = tf.argmax(out, axis=1)
truth = tf.argmax(y, axis=1)
print('预测类别:', pred[:10].numpy())
print('真实类别:', truth[:10].numpy())

predict x: (128, 784)
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
out: (128, 10)
[[-2.0951002e+00  4.7759042e+00  6.2473502e+00  6.1024351e+00
   5.0671892e+00 -4.5136204e+00 -1.5669029e+01  1.9422318e+01
  -3.3091502e+00  3.2551885e+00]
 [ 2.5257378e+00  2.3190951e+00  7.1797938e+00  2.3401811e+00
  -2.5323588e-01 -4.4774442e+00 -4.7035715e-01 -1.5520930e-02
   1.2274853e+00 -5.7152267e+00]
 [-1.0248353e+01  1.5140892e+01 -3.9559951e+00 -1.1152079e+01
   5.6431562e-01 -3.4064560e+00  1.3162628e+00  1.5502508e-01
  -6.0379064e-01  6.1951500e-01]]
预测类别: [7 2 1 0 4 1 4 9 6 9]
真实类别: [7 2 1 0 4 1 4 9 5 9]


如果只是测试模型在整个测试集上的性能，可以使用 `Model.evaluate(db)`。它会遍历
传入的数据集，并按照 `compile()` 中配置的损失函数和测量指标统计结果。

In [7]:
# 模型测试，评估在 test_db 上的性能表现
test_loss, test_acc = network.evaluate(test_db)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 611us/step - accuracy: 0.9676 - loss: 0.1198
Test loss: 0.11975512653589249
Test accuracy: 0.9675999879837036
